# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates loading and exploring the **FAIR²** dataset for clinicopathological and molecular characteristics of second primary colorectal cancer in cancer survivors, using the `mlcroissant` library. The notebook follows a reproducible template for Croissant-based datasets.

### Dataset Source
The dataset source is provided via the Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure mlcroissant is installed (uncomment below if needed in your environment)
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)
# Access dataset metadata (as an object, not a dict)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets and their fields. All entities (record sets, fields, columns) are referenced via their `@id`.

In [ ]:
# List all record sets by their @id and name for later reference

def list_record_sets(ds):
    print("Available Record Sets:")
    for rs in ds.record_sets:
        print(f"- @id: {rs.id}\n  name: {rs.name}")
        # List fields in each record set with their @id
        print("  Fields:")
        for f in rs.fields:
            print(f"    - @id: {f.id} | name: {f.name} | dataType: {getattr(f, 'data_type', 'N/A')}")
        print()

list_record_sets(dataset)

## 3. Data Extraction
Load data from record sets using their `@id`.
We'll extract dataframes for each main record set found above.


In [ ]:
# Prepare to extract all top-level record sets to pandas DataFrames

record_sets = [rs.id for rs in dataset.record_sets]
if not record_sets:
    print("No record sets found in this Croissant package.")
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records from RecordSet with @id: {record_set_id}")
    recs = list(dataset.records(record_set=record_set_id))
    if recs:
        dataframes[record_set_id] = pd.DataFrame(recs)
        print(f"Loaded {len(recs)} records, columns: {dataframes[record_set_id].columns.tolist()}")
    else:
        print(f"No records found for @id {record_set_id}.")
    print()

# For demonstration, pick the first record set loaded (if any) for further analysis
if dataframes:
    main_record_set_id = next(iter(dataframes))
    print(f"Using record set @id: {main_record_set_id} for EDA.")
    print("Sample rows:")
    display(dataframes[main_record_set_id].head())


## 4. Exploratory Data Analysis (EDA)

Demonstrate typical data operations using record set and field `@id`s. We'll filter records, normalize numeric fields, and group by a categorical field. All variables reference the `@id` as found above.


In [ ]:
# EDA: Find numeric and categorical fields by their @id

import numpy as np

# Identify numeric and categorical fields in the main record set
main_rs = None
for rs in dataset.record_sets:
    if rs.id == main_record_set_id:
        main_rs = rs
        break
if not main_rs:
    raise ValueError("Could not find main RecordSet metadata.")

# List field @id and dataType for reference
numeric_field_id = None
group_field_id = None
for f in main_rs.fields:
    dtype = getattr(f, 'data_type', None)
    if dtype in ('Integer', 'Float', 'Number') and not numeric_field_id:
        numeric_field_id = f.id
    if dtype in ('Text', 'String') and not group_field_id:
        group_field_id = f.id
    if numeric_field_id and group_field_id:
        break

print(f"Selected numeric_field_id: {numeric_field_id}")
print(f"Selected group_field_id: {group_field_id}")

df = dataframes[main_record_set_id]

if numeric_field_id in df.columns:
    # Coerce to numeric, errors to NaN
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    # Example: Threshold filter (if data non-empty and numeric)
    threshold = df[numeric_field_id].median() or 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping (if group field exists)
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
        display(grouped_df.head())
else:
    print(f"No numeric field found for field @id {numeric_field_id}.")

## 5. Visualization

Visualize data distributions or relationships between fields referencing all columns using their `@id`.

In [ ]:
# Example visualization: histogram of selected numeric field
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns and df[numeric_field_id].notnull().any():
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # If grouped_df is available, plot group means
    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(10, 5))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
        plt.title(f'Mean {numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()


## 6. Conclusion

In this notebook, we demonstrated how to load and explore a Croissant-formatted dataset (FAIR²) using the `mlcroissant` library. Key steps included reviewing record set and field `@id`s, extracting data frames using those identifiers, performing EDA and normalization of numeric fields, grouping, and data visualization. This workflow ensures reproducibility and transparency when working with FAIR data standards.